In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import argparse
import copy
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.distributions as TD
import torchvision.transforms as tr
from PIL import Image
from torchvision import datasets
from tqdm import tqdm

import wandb
from configs.energy_based.model import EBMConfig
from mnist2to3.utils import plot_diagnostics, plot_images, steps_counter
from src.costs.convolutional import NonlocalCost, UNetCost, VanillaCost
from src.costs.lse import MLPLSECost
from src.costs.mlp_based import MLPCost
from src.costs.nonlearnable import SquareCost
from src.models.energy_based import EGEOT
from src.plotting.distributions import plot_swiss_roll
from src.potentials.mlp_based import MLPPotential
from src.potentials.vanilla import NonlocalPotential, VanillaPotential
from src.samplers.energy_based.sample_buffer import SampleBufferEgEOT
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.dataset.colored_mnist import (
    apply_random_color,
    download_digit_images,
    get_paired_digits,
)
from src.utils.discrete_ot import OTPlanSampler
from src.utils.paired import generate_paired_data, get_GT_points, get_paired_sampler
from src.utils.train import compute_loss, update_average

In [3]:
WANDB_PROJECT_NAME = "eot"
DISCRETE_OT_DIR = "../src/discreteot"
sys.path.append(DISCRETE_OT_DIR)
from src.discreteot import DiscreteEOT_l2sq

In [4]:
EXP_NAME = "mnist2to3_s500_pVanilla_h0.01_P200"
EXP_DIR = "./out_data/{}/".format(EXP_NAME)
# json file with experiment config
CONFIG_FILE = "./config_locker/{}.json".format(EXP_NAME)

FROM_ITERATION = 7000
EVAL = True

FULL_DEVICE = f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu"
print(FULL_DEVICE)
USE_WANDB = False

In [5]:
# load experiment config
with open(CONFIG_FILE) as file:
    config = json.load(file)

# make directory for saving results
os.makedirs(EXP_DIR, exist_ok=True)
for folder in ["checkpoints", "shortrun", "longrun", "plots", "code"]:
    # os.mkdir(EXP_DIR + folder, exist_ok=True)
    os.makedirs(EXP_DIR + folder, exist_ok=True)

In [6]:
# set seed for cpu and CUDA, get device
# DEVICE SETTING
if FULL_DEVICE.startswith("cuda"):
    device = "cuda"
    GPU_DEVICE = int(FULL_DEVICE.split(":")[1])
    torch.cuda.set_device(GPU_DEVICE)
else:
    device = "cpu"

torch.manual_seed(config["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

In [7]:
# torch.set_default_device(device)
dtype = torch.float64
torch.torch.set_default_dtype(dtype)

In [8]:
# torch.manual_seed(train_config.seed)
# np.random.seed(train_config.seed)
# random.seed(train_config.seed)

## 2. Training Setup

In [9]:
HREG = config["hreg"]
EMA_UPDATE = config["ema_update"]

In [10]:
# set up potential
potential_bank = {"vanilla": VanillaPotential, "nonlocal": NonlocalPotential}
f = potential_bank[config["potential_type"]](n_c=config["im_ch"]).to(device)
# set up optimizer
optim_bank = {"adam": torch.optim.Adam, "sgd": torch.optim.SGD}
if config["optimizer_type"] == "sgd" and config["epsilon"] > 0:
    # scale learning rate according to langevin noise for invariant tuning
    config["lr_init"] *= (config["epsilon"] ** 2) / 2
    config["lr_min"] *= (config["epsilon"] ** 2) / 2

In [11]:
# set up cost
cost_bank = {
    "vanilla": VanillaCost,
    "nonlocal": NonlocalCost,
    "unet": UNetCost,
}
cost = cost_bank[config["cost_type"]](n_c=config["im_ch"]).to(device)

In [12]:
model_config = EBMConfig()
model = EGEOT(
    potential=f,
    cost=cost,
    sample_buffer=None,
    config=model_config,
).to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=config["lr_init"])
if FROM_ITERATION > 0:
    model.load_state_dict(
        torch.load(
            Path(EXP_DIR) / "checkpoints" / f"model_{FROM_ITERATION:>06d}.pth", weights_only=True, map_location=device
        )
    )
    optimizer.load_state_dict(
        torch.load(
            Path(EXP_DIR) / "checkpoints" / f"optim_{FROM_ITERATION:>06d}.pth", weights_only=True, map_location=device
        )
    )

if EMA_UPDATE:
    model_copy = copy.deepcopy(model)

In [16]:
SOURCE_DATASET = "MNIST"
TARGET_DATASET = "MNIST"
SOURCE_DIGIT = 2
TARGET_DIGIT = 3
P_XY_PAIRED_SAMPLES = config["P_XY"]

In [17]:
source_images: list[torch.Tensor] = download_digit_images(SOURCE_DATASET, SOURCE_DIGIT, 10000)
target_images: list[torch.Tensor] = download_digit_images(TARGET_DATASET, TARGET_DIGIT, 20000)

q_x_paired, q_y_paired = get_paired_digits(
    source_images, target_images, P_XY_PAIRED_SAMPLES, hue_offset=120, device=device
)

q_x = torch.stack([apply_random_color(digit, 360 * torch.rand(1)) for digit in source_images]).to(device)
q_y = torch.stack([apply_random_color(digit, 360 * torch.rand(1)) for digit in target_images]).to(device)

print(f"P_XY_PAIRED: {q_x_paired.shape}; Q_X_UNPAIRED: {q_x.shape}; R_Y_UNPAIRED: {q_y.shape}")

In [18]:
# initialize persistent images from noise (one persistent image for each data image)
# s_t_0 is used when init_type == 'persistent' in sample_s_t()
s_t_0 = 2 * torch.rand_like(q_x) - 1

In [19]:
# sample batch from given array of images
def sample_image_set(image_set: torch.Tensor, size: int = config["batch_size"]):
    rand_inds = torch.randperm(image_set.shape[0])[0:size]
    return image_set[rand_inds], rand_inds


################# DOT for init
def solve_dot(X: torch.Tensor, Y: torch.Tensor, numitermax: int = 10000, verbose: bool = False):
    DOT_DTYPE = "torch64"
    DOT_NUMITERMAX = numitermax
    DOT_VERBOSE = verbose
    discr_eot = DiscreteEOT_l2sq(device=device, verbose=DOT_VERBOSE, numItermax=DOT_NUMITERMAX, dtype=DOT_DTYPE).solve(
        X.view(X.size(0), -1), Y.view(Y.size(0), -1), HREG
    )
    x_inds = torch.arange(X.size(0))
    y_inds = discr_eot.sample_by_indices(x_inds, return_indices=True)
    y_image_subset = Y[y_inds]
    return X, y_image_subset, (x_inds, y_inds)


if config["shortrun_init"] == "persistentDOT":
    SC = steps_counter(s0=config["pDOT_update_step"], s1=1)

## 3. Functions for Sampling

In [20]:
# sample positive images from dataset distribution q_y (add noise to ensure min sd is at least langevin noise sd)
def sample_q_y():
    x_q_y = sample_image_set(q_y)[0]
    return x_q_y + config["data_epsilon"] * torch.randn_like(x_q_y)


def sample_pairs():
    x, inds = sample_image_set(q_x_paired)
    y = q_y_paired[inds]
    return x + config["data_epsilon"] * torch.randn_like(x), y + config["data_epsilon"] * torch.randn_like(y)


In [21]:
# get initial mcmc states for langevin updates ("persistent", "data", "uniform", or "gaussian")
def sample_s_t_0(init_type) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    returns (y_samples, x_samples, indices)
    """
    if init_type == "persistent":
        y_image_subset, rand_inds = sample_image_set(s_t_0)
        return y_image_subset, q_x[rand_inds], rand_inds
    elif init_type == "DOT":
        X, _ = sample_image_set(q_x)
        Y, _ = sample_image_set(q_y)
        x_image_subset, y_image_subset, _ = solve_dot(X, Y, numitermax=2000)
        return y_image_subset, x_image_subset, None
    elif init_type == "source_data":
        x_image_subset, inds = sample_image_set(q_x)
        return x_image_subset.clone().detach(), x_image_subset, (inds,)
    elif init_type == "target_data":
        x_image_subset, inds = sample_image_set(q_x)
        y_image_subset, y_inds = sample_image_set(q_y)
        return y_image_subset, x_image_subset, (inds, y_inds)
    elif init_type == "persistentDOT":
        y_image_subset, rand_inds = sample_image_set(s_t_0)
        if next(SC):
            X, x_inds = sample_image_set(q_x, size=1000)
            Y, _ = sample_image_set(q_y, size=1000)
            X, Y_dot, _ = solve_dot(X, Y, numitermax=10000)
            s_t_0[x_inds] = Y_dot
        return y_image_subset, q_x[rand_inds], rand_inds
    elif init_type == "uniform":
        x_image_subset, _ = sample_image_set(q_x)
        noise_image = 2 * torch.rand([config["batch_size"], config["im_ch"], config["im_sz"], config["im_sz"]]) - 1
        return noise_image.to(device), x_image_subset, None
    elif init_type == "gaussian":
        x_image_subset, _ = sample_image_set(q_x)
        noise_image = torch.randn([config["batch_size"], config["im_ch"], config["im_sz"], config["im_sz"]])
        return noise_image.to(device), x_image_subset, None
    elif init_type == "from_cost":
        x_image_subset, _ = sample_image_set(q_x)
        y_s_t = model.cost.net(x_image_subset)
        return y_s_t, x_image_subset, None
    else:
        raise RuntimeError('Invalid method for "init_type" (use "persistent", "data", "uniform", or "gaussian")')

In [22]:
# initialize and update images with langevin dynamics to obtain samples from finite-step MCMC distribution s_t
# TODO: update_s_t_0 seems to be buffer?
def sample_s_t(
    model: EGEOT,
    y_s_t_0: torch.Tensor,
    x_s_t_0: torch.Tensor,
    num_steps: int,
    init_type: str,
    s_t_0_inds: torch.Tensor | None = None,
    update_s_t_0: bool = True,
):
    # initialize MCMC samples
    # y_s_t_0, x_s_t_0, s_t_0_inds = sample_s_t_0()

    # iterative langevin updates of MCMC samples
    r_s_t = torch.zeros(1).to(device)  # variable r_s_t (Section 3.2) to record average gradient magnitude
    cost_grad_s_t = torch.zeros(1).to(device)
    for _ in tqdm(range(num_steps), leave=False):
        f_prime = model.potential.grad_y(y_s_t_0)
        cost_grad = model.cost.grad_y(x_s_t_0, y_s_t_0)
        y_s_t_0 += (f_prime - cost_grad) / (2 * HREG) + config["epsilon"] * torch.randn_like(y_s_t_0)
        r_s_t += f_prime.view(f_prime.shape[0], -1).norm(dim=1).mean()
        cost_grad_s_t += cost_grad.view(f_prime.shape[0], -1).norm(dim=1).mean()

    if init_type == "persistent" and update_s_t_0:
        # update persistent image bank
        s_t_0.data[s_t_0_inds] = y_s_t_0.detach().data.clone()

    return y_s_t_0.detach(), x_s_t_0, r_s_t.squeeze() / num_steps, cost_grad_s_t.squeeze() / num_steps

## 4. Training

In [23]:
# containers for diagnostic records (see Section 3)
d_s_t_record = torch.zeros(config["num_train_iters"]).to(
    device
)  # energy difference between positive and negative samples
r_s_t_record = torch.zeros(config["num_train_iters"]).to(
    device
)  # average image gradient magnitude along Langevin path

In [24]:
if USE_WANDB:
    wandb.init(name=EXP_NAME, project=WANDB_PROJECT_NAME, reinit=True, config=config)
    print("WandB has initialized.")

In [25]:
if EVAL:
    print("Evaluation has started.")
    print(
        "{:>6d}   Generating long-run samples. (L={:>6d} MCMC steps)".format(
            FROM_ITERATION + 1, config["num_longrun_steps"]
        )
    )
    N_PER = 6
    NUM_EVAL_SAMPLES = 3
    assert NUM_EVAL_SAMPLES > 0
    _step = torch.tensor(360 / NUM_EVAL_SAMPLES)

    _q_x_eval, _ = sample_image_set(torch.stack(source_images), NUM_EVAL_SAMPLES)
    q_x_eval = torch.stack([apply_random_color(_q_x_eval[i], _step * i) for i in range(NUM_EVAL_SAMPLES)]).to(device)

    _x_s_t_0 = q_x_eval.repeat_interleave(N_PER, 0)
    y_s_t_0, x_s_t_0 = _x_s_t_0.clone(), _x_s_t_0
    for init_type in [config["shortrun_init"]]:  # ["DOT", "persistent", "uniform", "source_data", "target_data"]:
        with torch.no_grad():
            _model = model_copy if EMA_UPDATE else model
            y_p_theta, x_p_theta, _, _ = sample_s_t(
                _model,
                y_s_t_0,
                x_s_t_0,
                num_steps=config["num_longrun_steps"],
                init_type=init_type,
                update_s_t_0=False,
            )

        plot_images(
            f"{init_type} init",
            y_p_theta,
            step=FROM_ITERATION + 1,
            save_dir=Path(EXP_DIR) / "longrun",
        )
        torch.save(y_p_theta, Path(EXP_DIR) / "longrun" / f"{init_type} init.pt")
        print("{:>6d}   Long-run samples for init {} saved.".format(FROM_ITERATION + 1, init_type))

## Plotting

In [26]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from torchvision.transforms import functional, Normalize
from mnist2to3.utils import tensor2image

In [27]:
init_type = "target_data"
N_PER = 5

In [31]:
y_200 = torch.load(
    Path("out_data/mnist2to3_s500_pVanilla_h0.01_P200/longrun/") / f"{init_type} init.pt",
    weights_only=True,
    map_location=device,
)
y_10 = torch.load(
    Path("out_data/mnist2to3_s500_pVanilla_h0.01_P10/longrun/") / f"{init_type} init.pt",
    weights_only=True,
    map_location=device,
)

In [32]:
x_200 = torch.load(
    Path("out_data/mnist2to3_s500_pVanilla_h0.01_P200/longrun/") / f"x.pt", weights_only=True, map_location=device
)
# x_10 = torch.load(
#     Path("out_data/mnist2to3_s500_pVanilla_h0.01_P10/longrun/") / f"x.pt", weights_only=True, map_location=device
# )

In [33]:
plot_images(
    f"init",
    x_200[::N_PER, :, :, :],
    normalize=True,
    nrow=1,
    clamp=True,
    step=FROM_ITERATION + 1,
)

In [35]:
image_shape = y_200[0].shape
NUM_EVAL_SAMPLES = 20


final_images_indices = torch.tensor([1, 3, 4, 6, 7, 9, 11, 12, 13, 15]).to(device)
NUM_FINAL_SAMPLES = len(final_images_indices)

_step = torch.tensor(360 / NUM_EVAL_SAMPLES)
_q_y_eval, _ = sample_image_set(torch.stack(target_images), NUM_EVAL_SAMPLES)
q_y_eval = torch.stack([apply_random_color(_q_y_eval[i], _step * i + 120) for i in range(NUM_EVAL_SAMPLES)]).to(
    device
)

In [36]:
tensor1 = x_200[::N_PER, :, :, :][final_images_indices].view(NUM_FINAL_SAMPLES, 1, *image_shape)
tensor2 = y_10[::N_PER, :, :, :][final_images_indices].view(NUM_FINAL_SAMPLES, 1, *image_shape)
tensor3 = y_200[::N_PER, :, :, :][final_images_indices].view(NUM_FINAL_SAMPLES, 1, *image_shape)
tensor4 = q_y_eval[final_images_indices].view(NUM_FINAL_SAMPLES, 1, *image_shape) # functional.adjust_hue(tensor1, 1/3)

In [48]:
plot_images(
    f"target_data_init_final_plot",
    torch.cat([tensor1, tensor4, tensor2, tensor3], dim=1).view(-1, *image_shape),
    normalize=True,
    nrow=4,
    clamp=True,
    step=FROM_ITERATION + 1,
    save_dir=Path("./out_data")
)

In [45]:
final_tensor = torch.cat([tensor1, tensor4, tensor2, tensor3], dim=1) #.view(-1, *image_shape)
normalize_transorm = Normalize(mean=[0.5], std=[0.5])
# final_tensor = tensor2image(final_tensor)
# final_tensor = tensor2image(normalize_transorm(final_tensor))
# final_tensor = normalize_transorm(final_tensor) 
final_tensor.shape

In [72]:
# Prepare the grid for plotting
fig, axes = plt.subplots(10, 4, figsize=(6, 15))  # (10 rows, 4 columns)

# Adjust space between columns
plt.subplots_adjust(wspace=0.25, hspace=0.25)

# Column titles
column_titles = [
    r"$x_1 \sim \pi_{x}^*$",
    r"$y_2 \sim \pi^*(\cdot \vert x_2)$",
    r"$y_1 \sim \pi^\theta_{10}(\cdot \vert x_1)$",
    r"$y_1 \sim \pi^\theta_{200}(\cdot \vert x_1)$",
]

# Set the titles for the columns
for col in range(4):
    axes[0, col].set_title(column_titles[col], fontsize=14)

# Loop through each row
for i in range(10):
    for col in range(4):
        ax = axes[i, col]  # Select the subplot for the i-th row and col-th column
        ax.imshow(final_tensor[i, col].permute(1, 2, 0).cpu().numpy())  # Convert (C, H, W) to (H, W, C)
        ax.axis("off")  # Hide axis

# Tight layout to make sure there is no overlap
# plt.tight_layout()
# plt.savefig("./out_data/EMB_mnist2to3_vertical.pdf", bbox_inches='tight')
plt.savefig("./out_data/EMB_mnist2to3_vertical.png", bbox_inches='tight')
plt.show()

In [71]:
# Row titles
row_titles = [
    r"$x_1 \sim \pi_{x}^*$",
    r"$y_2 \sim \pi^*(\cdot \vert x_2)$",
    r"$y_1 \sim \pi^\theta_{10}(\cdot \vert x_1)$",
    r"$y_1 \sim \pi^\theta_{200}(\cdot \vert x_1)$",
]

# Create a figure with an extra column for titles
fig, axes = plt.subplots(4, 11, figsize=(16, 5))  # 11 columns now (10 for data + 1 for titles)

# Adjust spacing (important to make room for titles)
plt.subplots_adjust(wspace=0.05, hspace=0.05, left=0.1) #Added more adjustment on the left

for row in range(4):
    # Get the "dummy" subplot for the title
    title_ax = axes[row, 0]

    # Set the title using text() for more control
    title_ax.text(0.5, 0.5, row_titles[row], fontsize=14, ha='center', va='center', transform=title_ax.transAxes)

    # Turn off axis for the title subplot
    title_ax.axis("off")

    # Plot the data in the remaining subplots
    for col in range(10):
        ax = axes[row, col + 1]  # Shift by 1 to account for the title column
        if final_tensor.shape[2] == 1:
            ax.imshow(final_tensor[col, row, 0].cpu().numpy(), cmap='gray')
        else:
            ax.imshow(final_tensor[col, row].permute(1, 2, 0).cpu().numpy())
        ax.axis("off")

# plt.tight_layout()
# plt.savefig("./out_data/EMB_mnist2to3_horizontal.pdf", bbox_inches='tight')
plt.savefig("./out_data/EMB_mnist2to3_horizontal.png", bbox_inches='tight')
plt.show()

In [103]:
plot_images(
    f"init",
    x_200,
    normalize=True,
    clamp=True,
    step=FROM_ITERATION + 1,
)

In [26]:
plot_images(
    f"init",
    toy_p_theta,
    normalize=True,
    clamp=True,
    step=FROM_ITERATION + 1,
)

In [54]:
from torchvision import transforms
normalize = transforms.Normalize(mean=[0.5], std=[0.5])


In [55]:
normalized_image = normalize(y_p_theta)

In [56]:
normalized_image.max()

In [61]:
plot_images(
    f"init",
    y_p_theta,
    normalize=True,
    clamp=True,
    step=FROM_ITERATION + 1,
)

In [65]:
plot_images(
    f"init",
    x_s_t_0,
    normalize=True,
    step=FROM_ITERATION + 1,
)

In [ ]:
if not EVAL:
    print("Training has started.")
    for i in range(FROM_ITERATION, config["num_train_iters"]):
        # obtain positive and negative samples
        samp_q_y = sample_q_y()
        y_s_t_0, x_s_t_0, s_t_0_inds = sample_s_t_0(init_type)
        with torch.no_grad():
            y_s_t, x_s_t, r_s_t, cost_grad_s_t = sample_s_t(
                model,
                y_s_t_0,
                x_s_t_0,
                num_steps=config["num_shortrun_steps"],
                s_t_0_inds=s_t_0_inds,
                init_type=config["shortrun_init"],
            )

        # calculate ML computational loss d_s_t (Section 3) for data and shortrun samples
        d_s_t = -f(samp_q_y).mean() + f(y_s_t).mean()
        # Uncomment also lines in sample_s_t. Maybe scale at the end?
        if config["epsilon"] > 0:
            # scale loss with the langevin implementation
            d_s_t *= 2 / (config["epsilon"] ** 2)
        # stochastic gradient ML update for model weights
        optimizer.zero_grad()
        d_s_t.backward()

        q_x_p, q_y_p = sample_pairs()
        paired_loss = model.compute_paired_loss(q_x_p, q_y_p)["loss"]
        if config["epsilon"] > 0:
            # scale loss with the langevin implementation
            paired_loss *= 2 / (config["epsilon"] ** 2)

        paired_loss.backward()
        optimizer.step()

        if EMA_UPDATE:
            update_average(model_copy, model, 0.99)

        # record diagnostics
        d_s_t_record[i] = d_s_t.detach().data
        r_s_t_record[i] = r_s_t

        # anneal learning rate
        for lr_gp in optimizer.param_groups:
            lr_gp["lr"] = max(config["lr_min"], lr_gp["lr"] * config["lr_decay"])

        # update wandb data
        if USE_WANDB:
            res_dict = {
                "d_s_t": d_s_t.detach().data,
                "r_s_t": r_s_t,
                "cost": paired_loss.detach().data,
                "cost_grad_s_t": cost_grad_s_t,
            }
            wandb.log({"train": res_dict}, step=i)

        # print and save learning info
        if (i + 1) == 1 or (i + 1) % config["log_freq"] == 0:
            print(
                "{:>6d}   d_s_t={:>14.9f}   r_s_t={:>14.9f}    cost_grad_s_t={:>14.9f}".format(
                    i + 1, d_s_t.detach().data, r_s_t, cost_grad_s_t
                )
            )
            # visualize synthesized images
            if EMA_UPDATE:
                with torch.no_grad():
                    y_s_t_0, x_s_t_0, s_t_0_inds = sample_s_t_0(init_type)
                    y_s_t, x_s_t, r_s_t, cost_grad_s_t = sample_s_t(
                        model_copy,
                        y_s_t_0,
                        x_s_t_0,
                        num_steps=config["num_shortrun_steps"],
                        s_t_0_inds=s_t_0_inds,
                        init_type=config["shortrun_init"],
                    )
            pbuff_dict = plot_images(
                f"pairs x->y, pbuff init",
                x_s_t,
                target_tensor=y_s_t,
                step=i + 1,
                use_wandb=USE_WANDB,
                save_dir=Path(EXP_DIR) / "shortrun",
            )
            # WARNING: work only for unet potential
            cost_dict = plot_images(
                f"pairs x->g(x)",
                x_s_t,
                target_tensor=model.cost.net(x_s_t),
                step=i + 1,
                use_wandb=USE_WANDB,
                save_dir=Path(EXP_DIR) / "shortrun",
            )
            wandb.log(pbuff_dict | cost_dict, step=i)

            if config["shortrun_init"] == "persistent":
                plot_images(
                    "Ys from pbuff",
                    s_t_0[0 : config["batch_size"]],
                    step=i,
                    save_dir=EXP_DIR + "shortrun/" + "y_s_t_0_{:>06d}.png".format(i + 1),
                    use_wandb=USE_WANDB,
                )
            # save network weights
            torch.save(model.state_dict(), EXP_DIR + "checkpoints/" + "model_{:>06d}.pth".format(i + 1))
            # save optimizer weights
            torch.save(optimizer.state_dict(), EXP_DIR + "checkpoints/" + "optim_{:>06d}.pth".format(i + 1))
            # plot diagnostics for energy difference d_s_t and gradient magnitude r_t
            # if (i + 1) > 1:
            #     plot_diagnostics(i, d_s_t_record, r_s_t_record, EXP_DIR + "plots/")
            # torch.cuda.empty_cache()

        # sample longrun chains to diagnose model steady-state
        if config["log_longrun"] and (i + 1) % config["log_longrun_freq"] == 0:
            print(
                "{:>6d}   Generating long-run samples. (L={:>6d} MCMC steps)".format(
                    i + 1, config["num_longrun_steps"]
                )
            )
            for init_type in [
                config["shortrun_init"]
            ]:  # ["DOT", "persistent", "uniform", "source_data", "target_data"]:
                with torch.no_grad():
                    y_s_t_0, x_s_t_0, s_t_0_inds = sample_s_t_0(init_type)
                    _model = model_copy if EMA_UPDATE else model
                    y_p_theta, x_p_theta, _, _ = sample_s_t(
                        _model,
                        y_s_t_0,
                        x_s_t_0,
                        num_steps=config["num_longrun_steps"],
                        init_type=init_type,
                        s_t_0_inds=s_t_0_inds,
                        update_s_t_0=False,
                    )
                longrun_dict = plot_images(
                    f"pairs x->y, longrun, {init_type} init",
                    x_p_theta,
                    target_tensor=y_p_theta,
                    step=i + 1,
                    use_wandb=USE_WANDB,
                    save_dir=Path(EXP_DIR) / "longrun",
                )
                wandb.log(longrun_dict, step=i)
                print("{:>6d}   Long-run samples for init {} saved.".format(i + 1, init_type))

        # WARNING: To reduce memory leakage
        # del samp_q_y, y_s_t, x_s_t, r_s_t


## 2. Config

In [6]:
Q_X_UNPAIRED_SAMPLES = 5958
R_Y_UNPAIRED_SAMPLES = 6131
P_XY_PAIRED_SAMPLES = 2000
IMG_SIZE = 32


LR_PAIRED = 1e-5
LR_UNPAIRED = 3e-5
SAMPLING_NUM_ITER = 500
MAX_STEPS = 100000
COST_FUNCTION = "MLP"
PAIRED_BATCH_SIZE = 256
UNPAIRED_BATCH_SIZE = 256
SEED = 30

## 3. Create data and samplers

In [9]:
from src.utils.dataset.colored_mnist import (
    apply_random_color,
    download_digit_images,
    get_paired_digits,
)

In [10]:
SOURCE_DATASET = "MNIST"
TARGET_DATASET = "MNIST"
SOURCE_DIGIT = 2
TARGET_DIGIT = 3
P_XY_PAIRED_SAMPLES = 5000

source_images: list[torch.Tensor] = download_digit_images(SOURCE_DATASET, SOURCE_DIGIT, 1000)
target_images: list[torch.Tensor] = download_digit_images(TARGET_DATASET, TARGET_DIGIT, 2000)

q_x_paired, q_y_paired = get_paired_digits(
    source_images, target_images, P_XY_PAIRED_SAMPLES, hue_offset=120, device=device
)

q_x = torch.stack([apply_random_color(digit, 360 * torch.rand(1)) for digit in source_images]).to(device)
q_y = torch.stack([apply_random_color(digit, 360 * torch.rand(1)) for digit in target_images]).to(device)

In [11]:
import torchvision as tv
import matplotlib.pyplot as plt
from torchvision.transforms import functional
from mnist2to3.utils import plot_im_pairs

In [12]:
x = q_x[:10]
y = q_y[:10]
# to_draw = torch.clamp(torch.cat([x.unsqueeze(1), y.unsqueeze(1)], 1).view(-1, *(3, 32, 32)), -1.0, 1.0)
to_draw = torch.cat([x.unsqueeze(1), y.unsqueeze(1)], 1).view(-1, *(3, 32, 32))
SB_torch_grid = tv.utils.make_grid(to_draw)

In [41]:
log_dict = plot_im_pairs("", x, y, nrow=2, use_wandb=False)

In [19]:
image_tensor = q_x_paired[0].roll(1, dims=0).permute(1, 2, 0)
image_tensor = functional.adjust_hue(q_x_paired[0], 1/3).permute(1, 2, 0)

# Plot the image
plt.imshow(image_tensor.detach().cpu().numpy())
plt.axis('off')  # Turn off axes for better visualization
plt.show()

In [18]:
image_tensor = q_y_paired[0].permute(1, 2, 0)

# Plot the image
plt.imshow(image_tensor.detach().cpu().numpy())
plt.axis('off')  # Turn off axes for better visualization
plt.show()

In [17]:
pd_train_sampler = get_paired_sampler(
    paired_source_samples, paired_target_samples, train_config.paired_batch_size, dataset_config.P_XY_paired, device
)

In [18]:
if dataset_config.Q_X_unpaired > 0:
    usd_sampler = DatasetSampler(unpaired_source_samples, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(paired_source_samples, device=device)

if dataset_config.R_Y_unpaired > 0:
    utd_sampler = DatasetSampler(unpaired_target_samples, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(paired_target_samples, device=device)

## 4. Model initialization

In [20]:
from src.costs.convolutional import NonlocalCost, VanillaCost, UNetCost
from src.potentials.vanilla import NonlocalPotential, VanillaPotential

In [42]:
potential = VanillaPotential(n_f=IMG_SIZE) 

In [43]:
cost = UNetCost(n_c=3)

In [46]:
model = EGEOT(potential, cost, sample_buffer_instance, model_config)

In [47]:
# For EMA update
if train_config.ema_update:
    model_copy = EGEOT(potential, cost, sample_buffer_instance, model_config)

## 5. Optimizers initialization

In [68]:
from src.utils.dataset.colored_mnist import (
    apply_random_color,
    download_digit_images,
    get_paired_digits,
)

In [69]:
SOURCE_DATASET = "MNIST"
TARGET_DATASET = "MNIST"
SOURCE_DIGIT = 2
TARGET_DIGIT = 3

In [70]:
source_images[0].shape

In [73]:
import torchvision.transforms as transforms

In [100]:
transform = transforms.Compose(
    [
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)


data = datasets.MNIST(root="./data", train=True, transform=transform, download=True)


indices = [i for i, label in enumerate(data.targets) if label == 3]

In [101]:
print([data[i][0] for i in indices[:2]])

In [93]:
source_images[0].shape

In [94]:
source_images: list[torch.Tensor] = download_digit_images(SOURCE_DATASET, SOURCE_DIGIT)
target_images: list[torch.Tensor] = download_digit_images(TARGET_DATASET, TARGET_DIGIT)

p_sampler = get_paired_digits(source_images, target_images, P_XY_PAIRED_SAMPLES, device=device)

q_x = torch.stack([apply_random_color(digit, 360 * torch.rand(1, device=digit.device)) for digit in source_images]).to(device)
q_y = torch.stack([apply_random_color(digit, 360 * torch.rand(1, device=digit.device)) for digit in target_images]).to(device)

In [51]:
q_x.shape, q_y.shape

In [52]:
q_y[:64].shape

In [59]:
model.potential.n_c

In [95]:
model.potential.net(q_y[0])

In [96]:
model.potential.grad_y(q_y[:64])

In [20]:
D_opt_unpaired = torch.optim.Adam(model.potential.parameters(), **opt_unpaired_config.model_dump())

In [21]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [22]:
# TODO: refactor this config
EXP_NAME = (
    "EgEOT_ColoredMNIST_"
    + f"COST_FUNCTION_{COST_FUNCTION}_"
    + f"P_XY_PAIRED_{dataset_config.P_XY_paired}_"
    + f"Q_X_UNPAIRED_{dataset_config.Q_X_unpaired}_"
    + f"R_Y_UNPAIRED_{dataset_config.R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"SAMPLING_STEPS_{model_config.sampling.num_iterations}_"
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    COST_FUNCTION=COST_FUNCTION,
    X_DIM=dataset_config.x_dim,
    Y_DIM=dataset_config.y_dim,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [23]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [ ]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(0, MAX_STEPS)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X, Y, compute_stats=True)
    D_loss_unpaired = output_unpaired["loss"]

    wandb.log({f"Unpaired: Loss": D_loss_unpaired.item()}, step=step)
    wandb.log({f"Unpaired: \int f(y)": output_unpaired["int_potential"].item()}, step=step)
    wandb.log({f"Unpaired: \int\log Z": output_unpaired["int_log_Z"].item()}, step=step)
    wandb.log({f"Unpaired: -E(x, y)": output_unpaired["neg_energy_t"].item()}, step=step)
    wandb.log({f"Unpaired: c(x, y)": output_unpaired["cost_t"].item()}, step=step)
    wandb.log({f"Unpaired: f(y)": output_unpaired["potential_t"].item()}, step=step)
    wandb.log({f"Unpaired: noise": output_unpaired["noise"].item()}, step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)

    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    wandb.log({f"Paired: Loss": D_loss_paired.item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)

    if step % train_config.plot_every == 0:
        distr_dict = plot_samples(model, unpaired_source_samples, log=True)
        wandb.log(distr_dict)
        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"model_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{train_config.steps_to}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_to}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_to}.pt"))

wandb.finish()

## Plotting

In [31]:
plot_samples(model, unpaired_source_samples)